# 02 · Explore with pandas
Group, aggregate, and avoid the classic **non-additive measure** trap.

In [ ]:
import sys
sys.path.insert(0, "..")          # lets the notebook import common.py from python_practice/
from common import *              # load_operations, check, OUT, ...
import pandas as pd

df = load_operations()            # reads the dashboard API if it is running, else data_samples/operations_clean.csv

## Additive measures just sum
Revenue and cost can be summed over any dimension.

In [ ]:
df.groupby("category")[["revenue", "operational_cost", "profit"]].sum().round(0)

## Margin is NOT additive
Margin is a ratio. The right overall number is *total profit ÷ total revenue*. Averaging each row's margin gives small rows the same vote as big ones.

In [ ]:
right = df["profit"].sum() / df["revenue"].sum()
wrong = df["margin"].mean()
print(f"weighted margin (ratio of sums): {right:.4%}")
print(f"average of row margins:          {wrong:.4%}   <- looks plausible, but is a different number")

## Your turn
Compute the weighted margin **per category** as a pandas Series (index = category). Hint: sum profit and revenue per group first, then divide.

In [ ]:
margin_by_category = None

In [ ]:
truth = df.groupby("category")["profit"].sum() / df.groupby("category")["revenue"].sum()
ok = margin_by_category is not None and (margin_by_category.sort_index() - truth.sort_index()).abs().max() < 1e-9
check("weighted margin per category", ok, "g = df.groupby('category')[['profit','revenue']].sum(); g.profit / g.revenue")

## Time intelligence
`resample` turns daily rows into weeks or months. `pct_change` gives period-over-period growth.

In [ ]:
monthly = df.set_index("record_date")["revenue"].resample("MS").sum()
pd.DataFrame({"revenue": monthly.round(0), "mom_growth_%": (monthly.pct_change() * 100).round(1)})

## Your turn 2
Which weekday (`day_name`) has the highest **average daily revenue**? Remember: average of the *daily totals*, not of rows.

In [ ]:
best_weekday = None

In [ ]:
daily = df.groupby(["record_date", "day_name"])["revenue"].sum().reset_index()
check("best weekday", best_weekday == daily.groupby("day_name")["revenue"].mean().idxmax(), "sum per day first, then average by day_name")